In [1]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\ishaan karwayun\OneDrive - Symbiosis International University\Desktop\BITSOM\Retail_demand_and_inventory_planner\etl
['ETL.ipynb', 'ETL_reproducible.py']


In [2]:
import pandas as pd
#importing datasets
calendar_og = pd.read_csv('../Retail Demand Forecasting & Inventory Replenishment Planner/calendar.csv')
inventory_og = pd.read_csv('../Retail Demand Forecasting & Inventory Replenishment Planner/inventory_daily.csv')
purchase_orders_og = pd.read_csv('../Retail Demand Forecasting & Inventory Replenishment Planner/purchase_orders.csv')
sales_daily_og = pd.read_csv('../Retail Demand Forecasting & Inventory Replenishment Planner/sales_daily.csv')
stores_og = pd.read_csv('../Retail Demand Forecasting & Inventory Replenishment Planner/stores.csv')

In [3]:
products_og = pd.read_json('../Retail Demand Forecasting & Inventory Replenishment Planner/products.json')

In [4]:
#converting date to datetime
sales_daily_og['date'] = pd.to_datetime(sales_daily_og['date'])
inventory_og['date'] = pd.to_datetime(inventory_og['date'])
calendar_og['date'] = pd.to_datetime(calendar_og['date'])
purchase_orders_og['order_date'] = pd.to_datetime(purchase_orders_og['order_date'])
purchase_orders_og['expected_receipt_date'] = pd.to_datetime(purchase_orders_og['expected_receipt_date'])




In [5]:
sales_daily = sales_daily_og.copy()
inventory = inventory_og.copy()
calendar = calendar_og.copy()
products = products_og.copy()
purchase_orders = purchase_orders_og.copy()
stores = stores_og.copy()

In [6]:
#defining as of date for forecasting
label_forecasting_days = 28
last_date = sales_daily['date'].max()
as_of_date = last_date - pd.Timedelta(days = label_forecasting_days)


In [7]:
as_of_date

Timestamp('2026-01-02 00:00:00')

In [8]:
#feature start date
feature_date = as_of_date - pd.Timedelta(weeks=8)

In [9]:
feature_date

Timestamp('2025-11-07 00:00:00')

In [10]:
#standardizing datesets
stores['region'] = stores['region'].str.lower()
stores['store_size'] = stores['store_size'].str.lower()
products['category'] = products['category'].str.lower()


In [11]:
purchase_orders[purchase_orders.duplicated()]
purchase_orders = purchase_orders.drop_duplicates()

In [12]:
inventory.isnull().sum()
inventory['on_hand_close'] = inventory['on_hand_close'].fillna(0)

In [13]:
inventory.head()

,date,store_id,sku_id,on_hand_open,receipts_units,on_hand_close
0,2025-07-04,ST001,SKU0001,58,0,55.0
1,2025-07-04,ST001,SKU0002,56,0,52.0
2,2025-07-04,ST001,SKU0003,121,0,113.0
3,2025-07-04,ST001,SKU0004,123,0,113.0
4,2025-07-04,ST001,SKU0005,103,0,98.0


In [14]:
# Calculate the 99.85th percentile (to target that 0.15% spike mention)
threshold = sales_daily['true_demand_units'].quantile(0.9985)

# Clip the values so no single day has a massive distortion
sales_daily['true_demand_units'] = sales_daily['true_demand_units'].clip(upper=threshold)

print(f"Capped all demand units above {threshold}")

Capped all demand units above 46.0


In [15]:
sales_daily = sales_daily.drop_duplicates()
fact_sales_store_sku_daily = sales_daily.groupby(['date','store_id','sku_id']).agg({'units_sold':'sum',
                                                                                   'true_demand_units':'sum',
                                                                                   'stockout_censored_units':'sum',
                                                                                   'revenue':'sum',
                                                                                   'margin_proxy':'sum'}).reset_index()    

In [16]:
fact_sales_store_sku_daily = fact_sales_store_sku_daily.merge(calendar[['date','day_of_week','promo_flag','holiday_flag']],how='left',on = 'date')

In [17]:
#fact_inventory_store_sku_daily.csv
inventory['stockout_flag'] = (inventory['on_hand_close']==0).astype(int)


In [18]:
fact_sales = fact_sales_store_sku_daily.copy()
fact_sales = fact_sales.sort_values('date')
fact_sales['avg_28d_demand'] = fact_sales.groupby(['store_id','sku_id'])['true_demand_units'].transform(lambda x: x.rolling(window=28, min_periods=1).mean())

In [19]:
fact_inventory_store_sku_daily = inventory.merge(fact_sales[['date','store_id', 'sku_id','avg_28d_demand']],on = ['date', 'store_id', 'sku_id'],how = 'left')
fact_inventory_store_sku_daily['avg_28d_demand'] = fact_inventory_store_sku_daily['avg_28d_demand'].fillna(0)

In [20]:
import numpy as np
fact_inventory_store_sku_daily['days_of_cover'] = round(((fact_inventory_store_sku_daily['on_hand_close'])/(fact_inventory_store_sku_daily['avg_28d_demand'])),2)
fact_inventory_store_sku_daily['days_of_cover'] = fact_inventory_store_sku_daily['days_of_cover'].replace([np.inf, -np.inf], 999)

In [21]:
fact_inventory_store_sku_daily.head()

,date,store_id,sku_id,on_hand_open,receipts_units,on_hand_close,stockout_flag,avg_28d_demand,days_of_cover
0,2025-07-04,ST001,SKU0001,58,0,55.0,0,3.0,18.33
1,2025-07-04,ST001,SKU0002,56,0,52.0,0,4.0,13.00
2,2025-07-04,ST001,SKU0003,121,0,113.0,0,8.0,14.12
3,2025-07-04,ST001,SKU0004,123,0,113.0,0,10.0,11.30
4,2025-07-04,ST001,SKU0005,103,0,98.0,0,5.0,19.60


In [22]:
#replenishment plan
recent_data = fact_sales[fact_sales['date']>=last_date - pd.Timedelta(days=56)] 

In [23]:
replenishment_inputs_store_sku = recent_data.groupby(['store_id','sku_id','date'])['true_demand_units'].sum().reset_index()

In [24]:
replenishment_inputs_store_sku = replenishment_inputs_store_sku.groupby(['store_id','sku_id']).agg(
    avg_daily_demand = ('true_demand_units','mean')
    ,demand_std_dev = ('true_demand_units','std')
).reset_index()

In [25]:
purchase_orders_grp = purchase_orders.groupby(['store_id','sku_id'])['lead_time_days'].mean().reset_index()


In [26]:
purchase_orders_grp

,store_id,sku_id,lead_time_days
0,ST001,SKU0010,7.0
1,ST001,SKU0025,5.0
2,ST001,SKU0036,2.0
3,ST001,SKU0059,7.0
4,ST001,SKU0085,2.0
...,...,...,...
80,ST018,SKU0068,3.0
81,ST018,SKU0070,7.0
82,ST018,SKU0072,3.0
83,ST018,SKU0073,2.0


In [27]:
replenishment_inputs_store_sku = replenishment_inputs_store_sku.merge(purchase_orders_grp,on = ['store_id','sku_id'],how='left')

In [28]:
avg_sku = replenishment_inputs_store_sku.groupby('sku_id')['lead_time_days'].mean()
replenishment_inputs_store_sku['lead_time_days'] = replenishment_inputs_store_sku['lead_time_days'].fillna(replenishment_inputs_store_sku['sku_id'].map(avg_sku))

In [29]:
replenishment_inputs_store_sku.head()

,store_id,sku_id,avg_daily_demand,demand_std_dev,lead_time_days
0,ST001,SKU0001,3.649123,2.669211,NaN
1,ST001,SKU0002,4.000000,2.771024,NaN
2,ST001,SKU0003,7.491228,4.342986,6.0
3,ST001,SKU0004,8.333333,4.943731,2.0
4,ST001,SKU0005,6.403509,3.692945,NaN


In [30]:
products['category'].unique()

array(['personalcare', 'homecare', 'snacks', 'beverages', 'dairy',
       'grocery'], dtype=object)

In [31]:
categories_sl = {'personalcare':0.95,'homecare':0.94,'snacks':0.96,'beverages':0.97,'dairy':0.98,'grocery':0.99}
def shelf_life(shelf_life_days):
    if shelf_life_days <40:
        return -0.05
    elif shelf_life_days >40:
        return -0.02
    return 0
products['service_level_target'] = products['category'].map(categories_sl)
products['service_level_target'] = (products['service_level_target']+products['shelf_life_days'].apply(shelf_life)).clip(0.85, 0.99)

In [32]:
from scipy.stats import norm

products['z_score'] = norm.ppf(products['service_level_target'])

In [33]:
replenishment_inputs_store_sku = replenishment_inputs_store_sku.merge(products[['sku_id','category','service_level_target','z_score']],on ='sku_id',how = 'left')

In [34]:
global_avg = replenishment_inputs_store_sku['lead_time_days'].mean()

replenishment_inputs_store_sku['lead_time_days'] = replenishment_inputs_store_sku['lead_time_days'].fillna(global_avg)

In [35]:
import numpy as np
replenishment_inputs_store_sku['safety_stock'] = np.sqrt(replenishment_inputs_store_sku['lead_time_days'])*replenishment_inputs_store_sku['demand_std_dev']*replenishment_inputs_store_sku['z_score']

In [36]:
replenishment_inputs_store_sku['ROP'] = (replenishment_inputs_store_sku['avg_daily_demand']*(replenishment_inputs_store_sku['lead_time_days'])) + replenishment_inputs_store_sku['safety_stock']

In [37]:
latest_inventory =  (fact_inventory_store_sku_daily.sort_values('date').groupby(['store_id','sku_id']).tail(1))

In [38]:
latest_inventory.head()

,date,store_id,sku_id,on_hand_open,receipts_units,on_hand_close,stockout_flag,avg_28d_demand,days_of_cover
341277,2026-01-30,ST012,SKU0088,0,0,0.0,1,10.392857,0.0
341284,2026-01-30,ST013,SKU0005,0,0,0.0,1,6.142857,0.0
341283,2026-01-30,ST013,SKU0004,0,0,0.0,1,6.714286,0.0
341282,2026-01-30,ST013,SKU0003,0,0,0.0,1,6.357143,0.0
341281,2026-01-30,ST013,SKU0002,0,0,0.0,1,2.714286,0.0


In [39]:
replenishment_inputs_store_sku = replenishment_inputs_store_sku.merge(latest_inventory[['store_id','sku_id','on_hand_close']], on=['store_id','sku_id'],how = 'left')

In [40]:
review_days = replenishment_inputs_store_sku['lead_time_days'].mean()
replenishment_inputs_store_sku['target_stock'] = (replenishment_inputs_store_sku['avg_daily_demand']*review_days) + replenishment_inputs_store_sku['safety_stock']

replenishment_inputs_store_sku['ROQ'] = replenishment_inputs_store_sku['target_stock']-replenishment_inputs_store_sku['on_hand_close']

In [41]:
#saving the dataset
output_folder = '../curated_dts'
fact_sales_store_sku_daily.to_csv(f'{output_folder}/fact_sales_store_sku_daily.csv', index=False)
fact_inventory_store_sku_daily.to_csv(f'{output_folder}/fact_inventory_store_sku_daily.csv', index=False)
replenishment_inputs_store_sku.to_csv(f'{output_folder}/replenishment_inputs_store_sku.csv', index=False)

In [42]:
products.to_csv('C:\\Users\\ishaan karwayun\\OneDrive - Symbiosis International University\\Desktop\\BITSOM\\Retail_demand_and_inventory_planner\\Retail Demand Forecasting & Inventory Replenishment Planner\\products.csv', index=False)